# Fine-tune Wav2Vec2 (CTC) on a South African language — local RTX 3060

**Anti-collapse + speed pass.** Changes from the previous version, and why:

1. **Length-feasibility filter turned ON** (was commented out). Samples where
   the downsampled audio has fewer frames than the label needs are CTC-infeasible
   and push the model toward degenerate "predict blank everywhere" shortcuts.
2. **Phased encoder freezing.** Only the top `N` transformer layers (+ `lm_head`)
   train initially; the rest of the 24-layer encoder stays frozen. ~300M free
   parameters against ~200 examples was enough to collapse in under 40 steps —
   cutting trainable capacity buys you a more stable starting point. You can
   unfreeze more layers once training is stable and you've scaled up the dataset.
3. **Real warmup.** `warmup_ratio=0.1` over 39 total steps was ~4 steps — LR
   was essentially at full value immediately. Now sized relative to actual
   total steps.
4. **bf16 instead of fp16.** Ampere (RTX 3060) supports bf16 natively; it has
   fp32's exponent range so it doesn't need loss scaling and is less prone to
   the instability that fp16 can produce on CTC's often-huge unnormalized loss values.
5. **Speed**: `group_by_length=True` to cut padding waste (audio clips vary a lot
   in length — padding to the longest in a random batch wastes real compute),
   `optim="adamw_torch_fused"` for a faster fused CUDA optimizer step, and
   `dataloader_num_workers` set to use your CPU cores instead of blocking on I/O.

**Expected data layout**: same as before — HF hub dataset
`dsfsi-anv/za-african-next-voices-compressed`, config = target language,
`train` / `dev_test` / `dev` splits, `transcript` field.

## 1. Install dependencies

In [1]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa torchcodec

## 2. Check GPU

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM (GB): 12.5
bf16 supported: True


## 3. Config — edit these

In [3]:
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda
BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
OUTPUT_DIR = f"./wav2vec2-{LANGUAGE}"



# Phased freezing: only the top N transformer encoder layers are trainable
# (out of 24 in wav2vec2-large-xlsr-53), plus lm_head. Raise this once a
# small run looks stable and you've scaled the dataset up.
NUM_TRAINABLE_ENCODER_LAYERS = 4

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [4]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

/home/khotso/projects/MultilingualASR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Load and normalize data

In [5]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
)

In [6]:
def filter_by_duration(dataset_dict, min_seconds=5.0, max_seconds=10.0, splits=None, num_proc=1):
    """
    Filters the given DatasetDict in-place keeping examples with duration in [min_seconds, max_seconds].
    - dataset_dict: datasets.DatasetDict
    - splits: list of split names to filter (None => all splits)
    - num_proc: number of processes for `.filter()`
    """
    if splits is None:
        splits = list(dataset_dict.keys())

    def _in_range(example):
        d = example.get("duration")
        return d is not None and (min_seconds <= d <= max_seconds)

    for split in splits:
        dataset_dict[split] = dataset_dict[split].filter(_in_range, num_proc=num_proc)

    return dataset_dict

In [7]:
from collections import defaultdict
import math

def duration_buckets(durations, bucket_size=1):
    """
    Bucket audio durations into fixed-width bins.

    Args:
        durations (list/iterable of float): duration in seconds per sample.
        bucket_size (float): width of each bucket in seconds.

    Returns:
        dict: {bucket_start: count}, sorted by bucket_start.
    """
    buckets = defaultdict(int)
    for d in durations:
        bucket_start = math.floor(d / bucket_size) * bucket_size
        buckets[bucket_start] += 1
    return dict(sorted(buckets.items()))


def print_duration_buckets(dataset_dict, splits=("train", "dev"), bucket_size=1):
    for split in splits:
        print(split)
        durations = dataset_dict[split]["duration"]
        buckets = duration_buckets(durations, bucket_size)

        total_dur = sum(durations)
        print(f"total duration: {total_dur} seconds")
        print(f"total duration: {total_dur/60} minutes")
        print(f"total duration: {total_dur/3600} hours")
        print(f"total samples: {len(durations)}")
        print()

        for bucket_start, count in buckets.items():
            bucket_end = bucket_start + bucket_size
            pct = 100 * count / len(durations)
            print(f"  {bucket_start:>5.1f} - {bucket_end:>5.1f} sec: {count:>6} samples ({pct:5.1f}%)")
        print()

In [8]:
dataset_dict = filter_by_duration(dataset_dict, min_seconds=5, max_seconds=6, splits=["train","dev"], num_proc=2)

In [9]:
print_duration_buckets(dataset_dict, splits=["train", "dev"], bucket_size=1)


train
total duration: 44233.739976 seconds
total duration: 737.2289996 minutes
total duration: 12.287149993333333 hours
total samples: 8039

    5.0 -   6.0 sec:   8033 samples ( 99.9%)
    6.0 -   7.0 sec:      6 samples (  0.1%)

dev
total duration: 3476.545541 seconds
total duration: 57.94242568333333 minutes
total duration: 0.9657070947222222 hours
total samples: 633

    5.0 -   6.0 sec:    632 samples ( 99.8%)
    6.0 -   7.0 sec:      1 samples (  0.2%)



In [10]:

dataset_dict["train"] = dataset_dict["train"].select(range(1000))
dataset_dict["dev"] = dataset_dict["dev"].select(range(100))



In [11]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].cast_column("audio", Audio(sampling_rate=16000))

In [12]:
print_duration_buckets(dataset_dict, splits=["train", "dev"], bucket_size=1)

train
total duration: 5513.46281 seconds
total duration: 91.89104683333333 minutes
total duration: 1.5315174472222222 hours
total samples: 1000

    5.0 -   6.0 sec:   1000 samples (100.0%)

dev
total duration: 550.194611 seconds
total duration: 9.169910183333334 minutes
total duration: 0.1528318363888889 hours
total samples: 100

    5.0 -   6.0 sec:    100 samples (100.0%)



In [13]:
import random



In [14]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [15]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [16]:
print(len(dataset_dict['train']))

1000


In [17]:
for split in ["train", "dev", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)

In [18]:
print(len(dataset_dict['train']))

998


In [19]:
random.seed(42)
random_indices = random.sample(range(len(dataset_dict["train"])), 10)
print("Random indices:", random_indices)
for i, idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

Random indices: [654, 114, 25, 759, 281, 250, 228, 142, 754, 104]
1: Nokana ya polase ya bona, e simolola go omelela.
2: Lefapha la Thuto le agile sekȏlȏ se se pȏtlana fa thoko ga tsela.
3: Gompieno kêrêkê e ne e tletse tota.
4: Badiri ba galefile fa ba lemoga phokotsȏ ya tšhelête ya bone.
5: O ne gape a mo tsenyetsa metsi gore Mansfield a kgone go lema.
6: Malomagwe o rata go tshega fela o na le pelo e e bosula.
7: Nokana ya polase ya bona, e simolola go omelela.
8: Boatlhamô jwa khutlonnetsepa e, ke 400.
9: O ka kgona go koba letsogo ka sekgono.
10: Ba itumeletse kôpanyo ya dikakanyô tsa bo mme ba lekgotla.


In [20]:
for split in ["train", "dev","dev_test"]:
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

In [21]:
for i,idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

1: nokana ya polase ya bona e simolola go omelela
2: lefapha la thuto le agile sekȏlȏ se se pȏtlana fa thoko ga tsela
3: gompieno kêrêkê e ne e tletse tota
4: badiri ba galefile fa ba lemoga phokotsȏ ya tšhelête ya bone
5: o ne gape a mo tsenyetsa metsi gore mansfield a kgone go lema
6: malomagwe o rata go tshega fela o na le pelo e e bosula
7: nokana ya polase ya bona e simolola go omelela
8: boatlhamô jwa khutlonnetsepa e ke
9: o ka kgona go koba letsogo ka sekgono
10: ba itumeletse kôpanyo ya dikakanyô tsa bo mme ba lekgotla


## 6. Build vocabulary from your transcripts

In [22]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

Map: 100%|██████████| 100/100 [00:00<00:00, 42358.15 examples/s]

Vocab size: 33


{"'": 1,
 'a': 2,
 'b': 3,
 'c': 4,
 'd': 5,
 'e': 6,
 'f': 7,
 'g': 8,
 'h': 9,
 'i': 10,
 'j': 11,
 'k': 12,
 'l': 13,
 'm': 14,
 'n': 15,
 'o': 16,
 'p': 17,
 'q': 18,
 'r': 19,
 's': 20,
 't': 21,
 'u': 22,
 'v': 23,
 'w': 24,
 'y': 25,
 'ê': 26,
 'ô': 27,
 'š': 28,
 'ȇ': 29,
 'ȏ': 30,
 '|': 0,
 '[UNK]': 31,
 '[PAD]': 32}

## 7. Build processor

In [23]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 8. Preprocess audio + labels

Same as before, using `processor.tokenizer(...)` directly (no deprecated `as_target_processor()`).

In [24]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

In [25]:
test_sentence = dataset_dict["train"][0]["transcript"]
test_sentence

'letsôgô la tsala ya gagwe le kgaogile kwa ntweng ya masole'

In [26]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=2)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4 if you have CPU cores to spare
    )

In [27]:
encoded = processor(text=test_sentence).input_ids
decoded = processor.decode(encoded)

print(f"Original: {test_sentence}")
print(f"Decoded:  {decoded}")

Original: letsôgô la tsala ya gagwe le kgaogile kwa ntweng ya masole
Decoded:  letsôgô la tsala ya gagwe le kgaogile kwa ntweng ya masole


In [28]:
print("Vocab size:", len(processor.tokenizer))
print("Pad token:", processor.tokenizer.pad_token)
print("Pad ID:", processor.tokenizer.pad_token_id)

Vocab size: 35
Pad token: [PAD]
Pad ID: 32


In [29]:
print(processor.tokenizer.special_tokens_map)
print(processor.tokenizer.all_special_tokens)
print(processor.tokenizer.all_special_ids)

{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'word_delimiter_token': '|'}
['<s>', '</s>', '[UNK]', '[PAD]', '|']
[33, 34, 31, 32, 0]


## 9. CTC length-feasibility filter — now ON

This is the single most important fix here. Wav2Vec2-large-XLSR-53
downsamples audio by roughly 320x (5 conv layers with strides
multiplying to ~320). If a clip's frame count after downsampling is
close to or below its label length, CTC has no valid alignment to find —
the loss on that sample balloons and gradients get dragged toward a
degenerate shortcut. Filtering these out before training removes that
source of collapse pressure entirely.

In [30]:
def length_ok(batch):
    # rough CTC feasibility check: downsampled frames must exceed label length
    approx_frames = batch["input_length"] // 320
    return approx_frames > len(batch["labels"])

before_counts = {split: len(dataset_dict[split]) for split in ["train", "dev"]}

for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(length_ok, num_proc=2)

for split in ["train", "dev"]:
    print(f"{split}: {before_counts[split]} -> {len(dataset_dict[split])} after length filter")

train: 998 -> 998 after length filter
dev: 100 -> 100 after length filter


In [31]:
sample = dataset_dict["train"][0]
print(len(sample["input_values"]))
print(len(sample["labels"]))

95573
58


## 10. Data collator

In [32]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 11. Metrics (WER / CER)

In [33]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        "examples": {"prediction": pred_str[:3], "label": label_str[:3]}
    }

## 12. Load model — phased freezing

Instead of unfreezing the whole 24-layer encoder, only the top
`NUM_TRAINABLE_ENCODER_LAYERS` layers + `lm_head` are trainable. This
directly targets the collapse: far fewer free parameters means far less
room for the model to find a cheap degenerate shortcut before it's had
enough steps to learn real acoustic-to-character alignment. Widen this
once you see stable, non-degenerate predictions and have scaled up the
dataset size.

In [34]:
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attn_implementation="sdpa",
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen

# Freeze all encoder transformer layers, then selectively unfreeze the top N.
total_layers = len(model.wav2vec2.encoder.layers)
# for i, layer in enumerate(model.wav2vec2.encoder.layers):
#     requires_grad = i >= (total_layers - NUM_TRAINABLE_ENCODER_LAYERS)
#     for param in layer.parameters():
#         param.requires_grad = requires_grad

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Trainable encoder layers: top {NUM_TRAINABLE_ENCODER_LAYERS} of {total_layers}")

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 33000.77it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 311,264,419 / 315,474,595 (98.67%)
Trainable encoder layers: top 4 of 24


In [35]:
print("ctc_zero_infinity:", model.config.ctc_zero_infinity)
print("ctc_loss_reduction:", model.config.ctc_loss_reduction)

ctc_zero_infinity: True
ctc_loss_reduction: mean


## 13. Training arguments

- `bf16=True` (was `fp16`): more stable on Ampere, no loss-scaling needed.
- `warmup_ratio` raised and computed against realistic step counts — with
  phased freezing you have far fewer trainable params, so it's worth giving
  the optimizer a real ramp rather than ~4 steps.
- `group_by_length=True`: batches similar-length clips together, cutting
  wasted compute on padding — this is usually the single biggest local
  speed win for variable-length audio.
- `optim="adamw_torch_fused"`: fused CUDA AdamW kernel, meaningfully
  faster per step than the default eager implementation.
- `dataloader_num_workers`: overlaps data loading with GPU compute instead
  of blocking on it every step.

In [36]:
import math
import torch
from transformers import TrainingArguments

def create_training_args(
    output_dir,
    train_dataset,
    num_epochs,
    per_device_train_batch_size,
    per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    evals_per_epoch=1,
    logs_per_epoch=4,
    saves_per_epoch=1,
    warmup_ratio=0.10,
):
    """
    Create TrainingArguments with step-based intervals computed from epochs.

    Parameters
    ----------
    evals_per_epoch : int
        Number of evaluations per epoch.
    logs_per_epoch : int
        Number of logging events per epoch.
    saves_per_epoch : int
        Number of checkpoints per epoch.
    warmup_ratio : float
        Fraction of total optimizer steps used for warmup.
    """

    world_size = max(torch.cuda.device_count(), 1)

    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            per_device_train_batch_size
            * gradient_accumulation_steps
            * world_size
        )
    )

    total_steps = steps_per_epoch * num_epochs

    warmup_steps = max(1, int(total_steps * warmup_ratio))
    eval_steps = max(1, steps_per_epoch // evals_per_epoch)
    logging_steps = max(1, steps_per_epoch // logs_per_epoch)
    save_steps = max(1, steps_per_epoch // saves_per_epoch)
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()

    print(f"Steps/epoch : {steps_per_epoch}")
    print(f"Total steps : {total_steps}")
    print(f"Warmup      : {warmup_steps}")
    print(f"Eval steps  : {eval_steps}")
    print(f"Log steps   : {logging_steps}")
    print(f"Save steps  : {save_steps}")
    print(f"bf16        : {bf16}")
    print(f"fp16        : {fp16}")

    return TrainingArguments(
        output_dir=output_dir,
        save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
        load_best_model_at_end=False,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_steps=save_steps,
        logging_steps=logging_steps,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        num_train_epochs=num_epochs,
        bf16=bf16,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        length_column_name="input_length",
        train_sampling_strategy="group_by_length",
        dataloader_num_workers=0,
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
        report_to=[],
    )

In [37]:
# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
#     load_best_model_at_end=False,  # disables loading best model at end, turn on for prod
#     per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
#     per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     eval_strategy="steps",
#     eval_steps=10,
#     save_steps=600,
#     logging_steps=10,
#     learning_rate=5e-5,
#     warmup_steps=10
#     num_train_epochs=3,
#     bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
#     fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
#     max_grad_norm=1.0,
#     gradient_checkpointing=True,  # trade speed for VRAM headroom
#     optim="adamw_torch_fused",
#     dataloader_num_workers=4,
#     save_total_limit=2,
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=False,
#     report_to=[],
# )
# RTX 3060 12GB: batch 4-8 with grad accumulation is a safe start.
# If you hit CUDA OOM, drop per_device_train_batch_size to 2-4.
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 2
PER_DEVICE_EVAL_BATCH = 4

training_args = create_training_args(
    output_dir=OUTPUT_DIR,
    train_dataset=dataset_dict["train"],
    num_epochs=0,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

Steps/epoch : 125
Total steps : 0
Warmup      : 1
Eval steps  : 125
Log steps   : 31
Save steps  : 125
bf16        : True
fp16        : False


In [38]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
)

## 14. Train

Watch the `examples` field in eval output for the first 2-3 evals — if
predictions are still degenerating into a single repeated character, stop
and check (in order): whether the length filter above actually removed
samples (if it removed ~0, the collapse wasn't a length-feasibility issue
and the next lever is dropping `learning_rate` further or reducing
`NUM_TRAINABLE_ENCODER_LAYERS`); whether `bf16` is actually active (printed
in section 2); and whether the effective batch size
(`PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS`) is large enough relative to
dataset size — very small effective batches on a tiny dataset make early
training noisy in a way that can also nudge toward collapse.

In [39]:
#trainer.train()

In [40]:
#trainer.state.log_history

## 15. Save

In [41]:
# trainer.save_model(OUTPUT_DIR)
# processor.save_pretrained(OUTPUT_DIR)
# print(f"Saved to {OUTPUT_DIR}")

In [42]:

processor = Wav2Vec2Processor.from_pretrained(OUTPUT_DIR)
model = Wav2Vec2ForCTC.from_pretrained(OUTPUT_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 10997.03it/s]


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

## 16. Quick sanity-check inference

In [43]:
import soundfile as sf

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

In [44]:
sample = dataset_dict["dev"].select(range(1))[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
print("Raw predicted IDs:", predicted_ids[0].tolist())
print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

Raw predicted IDs: [32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 10, 32, 32, 32, 32, 32, 32, 32, 28, 9, 6, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 15, 15, 6, 32, 32, 32, 0, 0, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 14, 14, 32, 32, 32, 14, 14, 6, 32, 0, 32, 32, 32, 32, 32, 20, 6, 6, 32, 32, 32, 32, 15, 15, 27, 32, 0, 0, 32, 32, 20, 20, 6, 32, 32, 32, 21, 13, 13, 6, 32, 32, 21, 20, 20, 2, 0, 0, 32, 21, 21, 9, 9, 9, 16, 32, 32, 32, 32, 32, 12, 12, 26, 32, 32, 32, 32, 32, 8, 8, 27, 32, 32, 0, 0, 25, 25, 2, 2, 0, 0, 32, 32, 32, 32, 32, 21, 6, 32, 32, 32, 32, 12, 12, 2, 2, 32, 32, 32, 32, 32, 21, 6, 6, 32, 32, 32, 32, 32, 32, 12, 2, 2, 2, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 15, 15, 27, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,

# Test 1: preprocessed dev split

In [45]:
def evaluate_split(split="dev_test", num_samples=5):
    print(f"=== Evaluation on {split} split ===\n")
    test_samples = dataset_dict[split].select(range(num_samples))

    pred_texts = []
    ref_texts = []

    for i, sample in enumerate(test_samples):
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.tokenizer.decode(predicted_ids[0])
        actual_text = processor.tokenizer.decode(sample["labels"], group_tokens=False)

        pred_texts.append(predicted_text)
        ref_texts.append(actual_text)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

    wer = wer_metric.compute(predictions=pred_texts, references=ref_texts)
    cer = cer_metric.compute(predictions=pred_texts, references=ref_texts)

    print(f"WER: {wer:.4f}")
    print(f"CER: {cer:.4f}")

    return {"wer": wer, "cer": cer, "predictions": pred_texts, "references": ref_texts}

In [46]:
evaluate_split(split="dev", num_samples=5)

=== Evaluation on dev split ===

--- Sample 1 ---
Predicted: išhene mme senô setletsa thokêgô ya tekatekanô
Actual:    išene mme seno se tlisa tlhokego ya tekatekano

--- Sample 2 ---
Predicted: mmelaô e mentshwa e ga gamatsa twantshô ya tšhebeben
Actual:    melao e mentšhwa e gagamatsa twantsho ya gbv

--- Sample 3 ---
Predicted: botlhopasa ba lemirui ba kwa tafelo kopkwa kobentalo
Actual:    setlhopha sa balemirui ba kwa tafelkop kwa groblersdal

--- Sample 4 ---
Predicted: a tshwanetse go o bamelamela wa natalo e la telang
Actual:    a tshwanetse go obamela melawanataolo e e latelang

--- Sample 5 ---
Predicted: momotshameki wa kgwele ya dinaôo o ne a konopa kgwele
Actual:    motshameki wa kgwele ya dinaô o ne a konopa kgwele

WER: 0.6190
CER: 0.1967


{'wer': 0.6190476190476191,
 'cer': 0.19672131147540983,
 'predictions': ['išhene mme senô setletsa thokêgô ya tekatekanô',
  'mmelaô e mentshwa e ga gamatsa twantshô ya tšhebeben',
  'botlhopasa ba lemirui ba kwa tafelo kopkwa kobentalo',
  'a tshwanetse go o bamelamela wa natalo e la telang',
  'momotshameki wa kgwele ya dinaôo o ne a konopa kgwele'],
 'references': ['išene mme seno se tlisa tlhokego ya tekatekano',
  'melao e mentšhwa e gagamatsa twantsho ya gbv',
  'setlhopha sa balemirui ba kwa tafelkop kwa groblersdal',
  'a tshwanetse go obamela melawanataolo e e latelang',
  'motshameki wa kgwele ya dinaô o ne a konopa kgwele']}

# Test 2: unprocessessed dev split

In [51]:
import torch

def evaluate_raw_split(split="dev_test", num_samples=5, print_n=5, normalize_fn=None):
    """
    Args:
        split (str): dataset split to evaluate on.
        num_samples (int): number of samples to evaluate (use len(dataset_dict[split]) for full split).
        print_n (int): number of sample predictions to print, regardless of num_samples.
        normalize_fn (callable, optional): function to apply to reference transcripts
            before WER/CER computation, to match training-time normalization.
    """
    print(f"=== Evaluation on {split} split ===\n")

    samples = dataset_dict[split].select(range(num_samples))

    pred_texts = []
    ref_texts = []

    model.eval()

    for i, sample in enumerate(samples):
        # Raw audio
        audio = sample["audio"]["array"]

        # Convert to model inputs
        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)

        # Decode prediction
        predicted_text = processor.batch_decode(predicted_ids)[0]

        # Reference transcription
        actual_text = sample["transcript"]  # <-- change if your transcript column has another name

        if normalize_fn is not None:
            actual_text = normalize_fn(actual_text)

        pred_texts.append(predicted_text)
        ref_texts.append(actual_text)

        if i < print_n:
            print(f"--- Sample {i+1} ---")
            print(f"Predicted: {predicted_text}")
            print(f"Actual:    {actual_text}\n")

    if num_samples > print_n:
        print(f"... ({num_samples - print_n} more samples evaluated but not printed)\n")

    wer = wer_metric.compute(
        predictions=pred_texts,
        references=ref_texts,
    )

    cer = cer_metric.compute(
        predictions=pred_texts,
        references=ref_texts,
    )

    print(f"WER: {wer:.4f}")
    print(f"CER: {cer:.4f}")

    return {
        "wer": wer,
        "cer": cer,
        "predictions": pred_texts,
        "references": ref_texts,
    }

In [ ]:
evaluate_raw_split(split="dev_test", num_samples=100, print_n=5)

=== Evaluation on dev_test split ===

--- Sample 1 ---
Predicted: taa a di tmogo thoa goa
Actual:    lenaane la tshegetso le tlhabololo ya balemirui

--- Sample 2 ---
Predicted: e ga ae   oo ya e  e  ora faaie e a e e e wa e a e o i o  a oke  a o  oa   gaeo  a ago g e aaooee poa ge a  ae e  aa aa
Actual:    leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona

--- Sample 3 ---
Predicted: agooa onoe nagaedi o
Actual:    dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse

--- Sample 4 ---
Predicted: mo o o gose o wa  kgoa e eengm a thoomonn  pa e  oe e we e kaooapang
Actual:    o ne a gana tšhono ya go dira kwa polaseng eo a neng rwala diratsuru kwa go yona go di rekisa

--- Sample 5 ---
Pred

[W804 14:16:31.593930281 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1207959552 bytes (free: 371785728, total: 12487294976).


... (195 more samples evaluated but not printed)

WER: 1.0022
CER: 0.7182


{'wer': 1.0022201665124884,
 'cer': 0.7181555586867691,
 'predictions': ['taa a di tmogo thoa goa',
  'e ga ae   oo ya e  e  ora faaie e a e e e wa e a e o i o  a oke  a o  oa   gaeo  a ago g e aaooee poa ge a  ae e  aa aa',
  'agooa onoe nagaedi o',
  'mo o o gose o wa  kgoa e eengm a thoomonn  pa e  oe e we e kaooapang',
  'aeee o  aaô goae e a me e se  ee e e le nng',
  'dwanggopo e awe yo ra oha ga ele eraka o  eo a goea',
  'aakgwe ra ile eirȏnoa o na o a o lele ene o ašhir šhaalmoraki eo o a  aeo oe a a o a a i e  opae e tšhap',
  'aao ra bo e phe eaaaa orae kga era aa o  e e   oa o naar',
  'naawapaoli ya tšhaa tšhanee o bagolo  e oraellô',
  'tle  oa ng ba hathamrao šha tšhao ra iri  a  ywatlapattarapa thng wnôtrrr tšhegwaog a renara  naga ra th ta t rar papyo ngwa ara aro a gogo o rrapan šha ethat  wangpa šhaktaô',
  'šarntata aa taetšh tta šeora  šhea o a oa ngwa ta rarraa ga ra aa tšhn te gwee a',
  'e ra nanga aga šae šhaloe tšh tšha  rwa o oe arabahnoara rra rara tšhagatšh

# Test 2: raw audio files (informal validation)

In [ ]:
import soundfile as sf
import torch
import torchaudio.transforms as T


def transcribe_audio(
    audio_path,
    model,
    processor,
    device,
    sampling_rate=16000,
    return_logits=False,
):
    model.eval()

    # Load using PySoundFile directly
    audio, native_sr = sf.read(audio_path, dtype="float32")

    # Convert stereo to mono if needed
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if native sampling rate differs from target sampling rate
    if native_sr != sampling_rate:
        audio_tensor = torch.from_numpy(audio).float().unsqueeze(0)
        resampler = T.Resample(orig_freq=native_sr, new_freq=sampling_rate)
        audio = resampler(audio_tensor).squeeze(0).numpy()

    # Feature extraction
    inputs = processor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True,
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    print(f"\nFile: {audio_path}")
    print(f"Prediction:\n{transcription}")

    if return_logits:
        return transcription, logits

    return transcription

In [ ]:
transcribe_audio(
    "rc4.wav",
    model,
    processor,
    device,
)

In [ ]:
transcribe_audio(
    "rc5.wav",
    model,
    processor,
    device,
)

In [ ]:
def transcribe_audio_file(filepath):
    audio, sr = sf.read(filepath)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.tokenizer.decode(predicted_ids[0])

In [ ]:
def test_on_raw_files(filepaths):
    print("=== Transcription on raw audio files ===\n")
    for path in filepaths:
        prediction = transcribe_audio_file(path)
        print(f"File: {path}")
        print(f"Predicted: {prediction}\n")

In [ ]:
raw_files = [
    "/home/khotso/data/validation_clips/clip1.wav",
    "/home/khotso/data/validation_clips/clip2.wav",
]
# test_on_raw_files(raw_files)